# Lilly v2 — train the reader (OCR on Kaggle)

Scope: `docs/V2-BOUNDARIES.md`. Latin Bosnian only. Generates synthetic crops on
Kaggle, trains 10 epochs, packages `lilly-read.zip`.

GPU + Internet on. Save & Run All.

In [ ]:
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path
import torch
assert torch.cuda.is_available(), "No GPU — enable GPU in session options"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(f"Cannot reach {url}: {exc}")
for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)

def run(*cmd):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)

In [ ]:
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Lilly"], check=True)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
os.chdir("/kaggle/working/Lilly")
print("cwd", os.getcwd())

In [ ]:
NEEDED = ["easyocr", "torch", "torchvision", "opencv-python-headless", "Pillow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])

In [ ]:
# Base reader weights (~easyocr latin_g2) from HF bundle
run("python3", "scripts/fetch_models.py")
assert Path("models/lilly/read/latin_g2.pth").is_file(), "fetch_models missing read weights"

In [ ]:
# Synthetic Bosnian crops — full v2 volume
run("python3", "data/scripts/generate_ocr_data.py", "--count", "20000", "--build-splits")
train_n = sum(1 for _ in open("data/ocr/train/gt.txt", encoding="utf-8"))
assert train_n > 5000, f"only {train_n} train crops"

In [ ]:
# Real human-labelled crops if present in clone
labels = Path("data/ocr/crops2/labels-human.tsv")
if labels.is_file():
    run("python3", "training/prepare_ocr_data.py", "--labels", str(labels))
    print("merged real crops")
else:
    print("no crops2 labels in repo — synthetic only")

In [ ]:
run("python3", "training/train_ocr.py", "--epochs", "10", "--batch-size", "32",
    "--keep-trained", "models/lilly/read-trained.pth")
trained = Path("models/lilly/read-trained.pth")
if trained.is_file():
    import shutil
    shutil.copy(trained, "models/lilly/read/lilly.pth")
    print("copied trained weights to lilly.pth for packaging")
assert Path("models/lilly/read/lilly.pth").is_file(), "no lilly.pth to ship"

In [ ]:
run("zip", "-qr", "/kaggle/working/lilly-read.zip", "models/lilly/read/lilly.pth",
    "models/lilly/read/lilly.yaml", "models/lilly/read/lilly.py")
size = Path("/kaggle/working/lilly-read.zip").stat().st_size
assert size > 100_000, f"zip too small: {size}"
print(f"lilly-read.zip — {size / 1048576:.1f} MB")